# How the annual PDs are calculated

The engine never reads a PD from its inputs. Every PD comes from one through-the-cycle (TTC) rating matrix per segment, shifted each year by that year's Z-score. This notebook rebuilds the PDs by hand from the input set, with numpy and scipy only, and checks every value against what a sealed engine run wrote.

"Annual PD" stands for three related numbers in the engine:

| Number | Formula | Used for | Engine table |
|---|---|---|---|
| One-year PD of grade g in year y | M(y)[g, D]: the D column of the year's point-in-time (PiT) matrix | the projection of the year: the share of each grade slice that defaults (R7) | `pit_matrices` |
| The yearly PDs of a curve, for a slice in grade g at the end of year y | F[k] = (e_g × M(y+1) × … × M(y+k))[D], then m[k], S[k] and q[k] | S1 and S2 ECL, Σ m[k] × EAD × LGD × DF (R8) | `pd_curves` |
| One-year portfolio PD | Σ GCA × q[1] ÷ Σ GCA over the performing slices | a parameter diagnostic (RPT-06) | `reports/portfolio_pd.csv` |

The sections follow the chain from the economy to the PD:

1. **Scenario → Z** (R4, SCN-08)
2. **Z after 2029** (R4, ECL-10)
3. **TTC matrix → PiT matrix → one-year PD** (R6, PD-10)
4. **One-year PDs → PD curve**, with rating migration (R6, PD-11, PD-03)
5. **The part-year bucket** of a maturity inside a year (R8, A7)
6. **One-year portfolio PD** (R10, RPT-06)
7. **Summary of the checks**

The notebook stands on its own: it imports no helper module and needs no other notebook. It reads the input set directly. For the comparison it opens the newest sealed run of that input set, and starts one if there is none. Use the project kernel (`.venv\Scripts\python.exe`, Python 3.12). The lifetime TTC PD behind the SICR line (MIG-07) is a separate calculation and is not covered here.

## Settings

The worked examples follow one segment, country, grade, scenario and valuation year-end. Change them and rerun: every check still covers the whole book.

`RUN_ID = "latest"` opens the newest sealed run of `INPUTS` with both scenarios, and starts one if there is none (about 30 seconds and a new folder under `runs/`). A run ID opens that run, and `None` always starts a new one. PDs do not depend on the variant: the variant changes repayment profiles and CCFs, and neither enters a PD.

In [ ]:
INPUTS = "inputs/514ed0e86e3dc760"  # the input set, relative to the repository root
RUN_ID = "latest"  # "latest", a run ID, or None to start a new run
SEGMENT = "P01"  # the worked examples: segment, country, grade, scenario, valuation year-end
COUNTRY = "NL"
GRADE = "G6"
SCENARIO = "adverse"
VALUATION = 2026

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
from matplotlib.ticker import PercentFormatter
from scipy.stats import norm

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
INPUT_DIR = ROOT / INPUTS
RUNS_DIR = ROOT / "runs"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)

# The colours of 01_full_run_walkthrough: each scenario keeps its colour in every chart.
COLOURS = {"baseline": "#2a78d6", "adverse": "#eb6834"}
INK, INK_SECONDARY, MUTED, GRID, AXIS, SURFACE = (
    "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "Helvetica", "Arial", "DejaVu Sans"],
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "semibold",
    "text.color": INK, "axes.labelcolor": INK_SECONDARY,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 2, "lines.solid_capstyle": "round",
    "legend.frameon": False, "legend.fontsize": 8.5,
})

## Inputs

Three files of the input set drive every PD: the parameter set (master scale, Z coefficients, ρ, horizon, terminal rule, tolerances), the scenario values and the TTC matrices. The parameter values are proposed, not owner-approved (assumption A2). The tolerance for every probability comparison below is the engine's own, from the parameter set.

In [ ]:
ps = yaml.safe_load((INPUT_DIR / "parameter_set.yaml").read_text(encoding="utf-8"))
scenario_doc = yaml.safe_load((INPUT_DIR / "scenarios.yaml").read_text(encoding="utf-8"))
ttc_table = pd.read_parquet(INPUT_DIR / "ttc_pd_matrix.parquet")

grades = list(ps["master_scale"]["grades"])  # performing grades, best first
labels = [*grades, ps["master_scale"]["default_grade"]]  # matrix rows and columns: G1..GN, D
n = len(grades)
rank = {g: i for i, g in enumerate(labels)}
years = [int(y) for y in ps["horizon"]["projection_years"]]
TOL = float(ps["tolerances"]["probability_identity"]["abs"])

print(f"{INPUTS}: parameter set {ps['parameter_set_id']} v{ps['version']} ({ps['status']}); "
      f"{n} grades plus D; projection years {years[0]}-{years[-1]}; "
      f"buckets to {ps['horizon']['last_bucket_year']}; probability tolerance {TOL:g}")
pd.DataFrame({s: {"portfolio": p["portfolio"], "rho": p["rho"], "Z drivers": ", ".join(p["z_drivers"])}
              for s, p in ps["segments"].items()}).T

## The engine run to compare with

Every number below is computed here first and then compared with the table the engine wrote. The run folder is only read, never written.

In [ ]:
def sealed_runs() -> pd.DataFrame:
    """Sealed runs of INPUTS with both scenarios, oldest first; blocked runs are left out."""
    rows = []
    for path in sorted(RUNS_DIR.glob("*/manifest.json")):
        m = json.loads(path.read_text(encoding="utf-8"))
        if (m["status"] != "BLOCKED" and m["input_set_hash"].startswith(INPUT_DIR.name)
                and {"baseline", "adverse"} <= set(m["config"]["scenarios"])):
            rows.append({"run_id": m["run_id"], "created_at": m["created_at"]})
    return pd.DataFrame(rows, columns=["run_id", "created_at"]).sort_values(["created_at", "run_id"])


def start_run() -> Path:
    """R1-R11 through stress.cli.execute, the entry point behind `uv run stress run`."""
    from stress import cli
    from stress.contracts.run import RunConfig

    config = RunConfig(inputs=INPUTS, scenarios=("baseline", "adverse"), variant="core")
    before = Path.cwd()
    os.chdir(ROOT)  # the engine resolves inputs/ against the working directory, as the CLI does
    try:
        ctx, sealed = cli.execute(config, RUNS_DIR)
    finally:
        os.chdir(before)
    if sealed["status"] == "BLOCKED":
        b = sealed["blocked"]
        raise RuntimeError(f"run {ctx.run_id} BLOCKED at {b['step']}: {b['code']}: {b['message']}")
    return Path(ctx.run_dir)


runs = sealed_runs()
if RUN_ID is None or (RUN_ID == "latest" and runs.empty):
    print("starting a new run, about 30 seconds")
    RUN_DIR = start_run()
else:
    RUN_DIR = RUNS_DIR / (runs.iloc[-1]["run_id"] if RUN_ID == "latest" else RUN_ID)
manifest = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
if not manifest["input_set_hash"].startswith(INPUT_DIR.name):
    raise ValueError(f"run {manifest['run_id']} used input set {manifest['input_set_hash'][:16]}, "
                     f"not {INPUT_DIR.name}")
if manifest["status"] == "BLOCKED":
    raise ValueError(f"run {manifest['run_id']} is BLOCKED: {manifest['blocked']}")
scenarios = list(manifest["config"]["scenarios"])


def engine(name: str) -> pd.DataFrame:
    """A table the run wrote: results/<name>.parquet, else results/reports/<name>.csv."""
    path = RUN_DIR / "results" / f"{name}.parquet"
    if path.is_file():
        return pd.read_parquet(path)
    return pd.read_csv(RUN_DIR / "results" / "reports" / f"{name}.csv")


CHECKS = []  # every comparison, for the summary in section 7


def compare(what: str, engine_values, here_values, limit: float = TOL) -> None:
    """Record and print the largest |a - b|; a missing value counts as a failure."""
    a = np.asarray(engine_values, dtype=float)
    b = np.asarray(here_values, dtype=float)
    gap = float(np.max(np.abs(a - b))) if len(a) and len(a) == len(b) else np.nan
    CHECKS.append({"check": what, "values": len(a), "largest gap": gap, "limit": limit})
    print(f"{what}: {len(a):,} values; largest gap {gap:.1e} (limit {limit:.0e})")


pd.Series({"run": manifest["run_id"], "status": manifest["status"],
           "variant": manifest["config"]["variant"], "scenarios": ", ".join(scenarios),
           "input set": manifest["input_set_hash"][:16], "code version": manifest["code_version"],
           "created": manifest["created_at"]}, name="engine run").to_frame()

## 1 · Scenario → Z (R4)

R4 combines each segment's macro drivers into one Z-score per segment, country, year and scenario (SCN-08):

$$Z = \sum_i \beta_i \times \frac{x_i - \mu_i}{\sigma_i}$$

The scenario file holds the values in percent. They are turned into decimals first (× 0.01), because μ and σ in the parameter set are decimals. β, μ and σ are set per segment and driver. A positive Z means a better economy than average, so drivers that hurt borrowers, such as unemployment and the reference rate, carry a negative β. The drivers of the worked segment and country:

In [ ]:
UNIT = {"percent": 0.01, "decimal": 1.0, "index": 1.0}  # scenario value units, as decimals
values = pd.DataFrame(scenario_doc["values"])
values["x"] = values["value"].astype(float) * values["unit"].map(UNIT)
assert values["x"].notna().all(), "a scenario value or unit is missing; the engine would block"

coefficients = pd.DataFrame([{"segment": s, "variable": v, **c}
                             for s, p in ps["segments"].items() for v, c in p["z_drivers"].items()])
in_years = values[values["year"].isin(years)][["scenario", "variable", "country", "year", "x"]]
contributions = coefficients.merge(in_years, on="variable")
contributions["contribution"] = (contributions["beta"] * (contributions["x"] - contributions["mean"])
                                 / contributions["sd"])
key = ["segment", "country", "scenario", "year"]
z_path = (contributions.groupby(key, as_index=False)["contribution"].sum()
          .rename(columns={"contribution": "z"}))

mine = contributions[(contributions["segment"] == SEGMENT) & (contributions["country"] == COUNTRY)]
print(f"Z of {SEGMENT} {COUNTRY}: x is the scenario value as a decimal")
display(mine.set_index(["scenario", "year", "variable"])[["x", "mean", "sd", "beta", "contribution"]]
        .sort_index())
z_path[(z_path["segment"] == SEGMENT) & (z_path["country"] == COUNTRY)].pivot(
    index="year", columns="scenario", values="z")

## 2 · Z after 2029 (R4)

A curve valued at end-2029 still needs a PD for every year to maturity, here up to the last bucket year. After the last scenario year the terminal rule (ECL-10) sets Z. Baseline stays flat at its 2029 value. Adverse reverts linearly to baseline 2029 over R years:

$$Z_{adv}(t) = a(t)\, Z_{adv}(2029) + \big(1 - a(t)\big)\, Z_{base}(2029), \qquad a(t) = \max\!\Big(0, \frac{R - (t - 2029)}{R}\Big)$$

With R = 6, a(t) is 5/6 in 2030 and reaches 0 in 2035, after which both scenarios hold the same flat Z. A missing year blocks the run. It is never set to zero (AT-41).

In [ ]:
rule = ps["terminal_rule"]
last, reversion = int(rule["last_scenario_year"]), int(rule["reversion_years"])
last_bucket = int(ps["horizon"]["last_bucket_year"])
at_last = z_path[z_path["year"] == last].pivot_table(index=["segment", "country"],
                                                     columns="scenario", values="z")
rows = []
for year in range(last + 1, last_bucket + 1):
    a = max(0.0, (reversion - (year - last)) / reversion)
    for (segment, country), z in at_last.iterrows():
        rows.append((segment, country, "baseline", year, z["baseline"]))
        rows.append((segment, country, "adverse", year, a * z["adverse"] + (1 - a) * z["baseline"]))
z_all = pd.concat([z_path, pd.DataFrame(rows, columns=key + ["z"])], ignore_index=True)
z_of = z_all.set_index(key)["z"]

path = z_all[(z_all["segment"] == SEGMENT) & (z_all["country"] == COUNTRY)
             & (z_all["year"] <= last + reversion + 1)].pivot(index="year", columns="scenario", values="z")
path["a(t)"] = [max(0.0, min(1.0, (reversion - (y - last)) / reversion)) for y in path.index]
print(f"Z of {SEGMENT} {COUNTRY}; flat from {last + reversion} to {last_bucket}")
display(path)

drivers = engine("drivers")
both = drivers.merge(z_all, on=key, how="outer", suffixes=("_engine", "_here"))
compare(f"R4 Z, {years[0]}-{last_bucket}", both["z_engine"], both["z_here"])

## 3 · TTC matrix → PiT matrix → one-year PD (R6)

Each segment has one one-year TTC matrix on the shared master scale (PD-09). A row is the grade at the start of a year and a column the grade at its end, and D is default. D is absorbing. The TTC matrix of the worked segment:

In [ ]:
def ttc_matrix(segment: str) -> np.ndarray:
    cells = ttc_table[ttc_table["segment"] == segment].pivot(
        index="from_grade", columns="to_grade", values="probability")
    return cells.reindex(index=labels, columns=labels).to_numpy()


TTC = {s: ttc_matrix(s) for s in ps["segments"]}
print(f"TTC matrix of {SEGMENT}")
pd.DataFrame(TTC[SEGMENT], index=labels, columns=labels).style.format("{:.3%}")

PD-10 shifts every performing row in cumulative form. C(i, j) is the probability of ending the year in grade j or worse from grade i, the row summed from the right, so C(i, G1) = 1 and C(i, D) = M(i, D). Each C is read as a threshold on a standard normal scale and shifted by the year's Z:

$$C_{PiT}(i,j) = \Phi\!\left(\Phi^{-1}\big(C_{TTC}(i,j)\big) - \sqrt{\rho}\, Z\right)$$

The PiT cells are the differences of neighbouring columns, M(i, j) = C_PiT(i, j) − C_PiT(i, j+1), and the D cell is C_PiT(i, D). C_PiT(i, G1) stays 1, so every row still sums to one. A lower Z raises every C_PiT, which moves probability toward worse grades and D. ρ is set per segment. Z = 0 gives back the TTC matrix exactly (D13). The D row is not shifted, so D stays absorbing.

The row of the worked grade in the first projection year, column by column. The G1 column has C = 1, whose threshold is +∞; it maps back to 1.

In [ ]:
rho = float(ps["segments"][SEGMENT]["rho"])
z_first = z_of[(SEGMENT, COUNTRY, SCENARIO, years[0])]
row = TTC[SEGMENT][rank[GRADE]]
c_ttc = row[::-1].cumsum()[::-1]
c_ttc[0] = 1.0
threshold = norm.ppf(c_ttc)
shifted = threshold - np.sqrt(rho) * z_first
c_pit = norm.cdf(shifted)
m_pit = c_pit - np.append(c_pit[1:], 0.0)

print(f"{SEGMENT} {COUNTRY}, {SCENARIO} {years[0]}: Z = {z_first:+.6f}, rho = {rho}; row {GRADE}")
steps = pd.DataFrame({"M_TTC": row, "C_TTC": c_ttc, "threshold = Φ⁻¹(C_TTC)": threshold,
                      "shifted = threshold − √ρ·Z": shifted, "C_PiT = Φ(shifted)": c_pit,
                      "M_PiT": m_pit}, index=labels).T
probabilities = pd.IndexSlice[["M_TTC", "C_TTC", "C_PiT = Φ(shifted)", "M_PiT"], :]
steps.style.format("{:.4f}").format("{:.4%}", subset=probabilities)

The same arithmetic for every row of every matrix. This builds one PiT matrix per segment, country, scenario and year from the first projection year to the last bucket year, from the Z of sections 1 and 2.

In [ ]:
def pit_matrix(ttc: np.ndarray, z: float, rho: float) -> np.ndarray:
    """PD-10: shift every performing row of a TTC matrix by Z; the D row stays absorbing."""
    c = ttc[:n, ::-1].cumsum(axis=1)[:, ::-1]  # C(i, j): grade j or worse
    c = c.clip(0.0, 1.0)  # guards against rounding in the sum
    c[:, 0] = 1.0  # G1 or worse is certain
    c_pit = norm.cdf(norm.ppf(c) - np.sqrt(rho) * z)
    c_pit[:, 0] = 1.0
    m = np.zeros_like(ttc)
    m[:n, :-1] = c_pit[:, :-1] - c_pit[:, 1:]
    m[:n, -1] = c_pit[:, -1]
    m[n, n] = 1.0
    return m


PIT = {(r.segment, r.country, r.scenario, int(r.year)):
       pit_matrix(TTC[r.segment], r.z, float(ps["segments"][r.segment]["rho"]))
       for r in z_all.itertuples()}
stack = np.stack(list(PIT.values()))
print(f"{len(PIT):,} PiT matrices built here, {years[0]}-{last_bucket}")
print(f"largest |row sum - 1|: {np.abs(stack.sum(axis=2) - 1).max():.1e}; "
      f"smallest cell: {stack.min():.1e}; "
      f"PD rises with the grade in every matrix: {bool((np.diff(stack[:, :n, -1], axis=1) > 0).all())}")

pit_engine = engine("pit_matrices")  # the engine reports the years up to the end of the reversion
here = [PIT[(r.segment, r.country, r.scenario, r.year)][rank[r.from_grade], rank[r.to_grade]]
        for r in pit_engine.itertuples()]
compare(f"R6 PiT matrix cells, {pit_engine['year'].min()}-{pit_engine['year'].max()}",
        pit_engine["probability"], here)

### The one-year PD

The D column of M(y) is each grade's one-year PD for year y (PD-10). It needs only the grade's own TTC PD, because C(g, D) = M_TTC(g, D):

$$PD_g(y) = \Phi\!\left(\Phi^{-1}(PD^{TTC}_g) - \sqrt{\rho}\, Z(y)\right)$$

Migration between performing grades does not change a one-year PD. It matters only from the second year of a curve (section 4). The one-year PDs of the worked segment and country:

In [ ]:
one_year = pd.DataFrame({"TTC": TTC[SEGMENT][:n, -1],
                         **{f"{s} {y}": PIT[(SEGMENT, COUNTRY, s, y)][:n, -1]
                            for s in scenarios for y in years}}, index=grades)
display(one_year.style.format("{:.3%}"))

closed, d_column = [], []
for (segment, country, scenario, year), m in PIT.items():
    r = float(ps["segments"][segment]["rho"])
    z = z_of[(segment, country, scenario, year)]
    closed.append(norm.cdf(norm.ppf(TTC[segment][:n, -1]) - np.sqrt(r) * z))
    d_column.append(m[:n, -1])
compare("PD-10: D column = one-grade formula, both built here", np.concatenate(d_column),
        np.concatenate(closed))

The one-year PD of the worked grade as a function of Z, with the scenario years marked. A lower Z gives a higher PD (AT-47). Z = 0 is an average year and gives back the TTC PD exactly, as the LGD and prepayment rules give back their TTC values (D13). Because the curve is convex, the PiT PD averaged over Z ~ N(0, 1) lies above the TTC PD: bad years add more PD than good years take away.

In [ ]:
pd_ttc = TTC[SEGMENT][rank[GRADE], -1]


def pd_of_z(z):
    return norm.cdf(norm.ppf(pd_ttc) - np.sqrt(rho) * z)


nodes, weights = np.polynomial.hermite_e.hermegauss(80)  # quadrature over Z ~ N(0, 1)
cycle_mean = (weights * pd_of_z(nodes)).sum() / weights.sum()
print(f"{GRADE} in {SEGMENT}: TTC PD {pd_ttc:.4%}; PiT PD at Z = 0 {pd_of_z(0.0):.4%}; "
      f"mean PiT PD over Z ~ N(0, 1) {cycle_mean:.4%}")

grid = np.linspace(-3, 3, 241)
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.plot(grid, pd_of_z(grid), color=INK_SECONDARY, linewidth=1.5, label=f"one-year PD of {GRADE}")
ax.axhline(pd_ttc, color=MUTED, linewidth=1, linestyle=(0, (4, 3)), label="TTC PD")
for s in scenarios:
    zs = np.array([z_of[(SEGMENT, COUNTRY, s, y)] for y in years])
    ax.scatter(zs, pd_of_z(zs), s=40, color=COLOURS[s], edgecolor=SURFACE, linewidth=1.5,
               zorder=3, label=f"{s} {years[0]}-{years[-1]}")
    if s != "baseline":  # label the stressed years; years with almost the same Z share a label
        clusters = []
        for i in np.argsort(zs):
            if clusters and zs[i] - zs[clusters[-1][-1]] < 0.15:
                clusters[-1].append(i)
            else:
                clusters.append([i])
        for cluster in clusters:
            z = zs[cluster].mean()
            ax.annotate(", ".join(str(years[i]) for i in sorted(cluster)), (z, pd_of_z(z)),
                        textcoords="offset points", xytext=(8, 3), fontsize=8, color=INK_SECONDARY)
ax.axvline(0, color=AXIS, linewidth=0.8)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Z (below 0: a worse economy than average)")
ax.set_title(f"One-year PD of {GRADE} against Z, {SEGMENT} (ρ = {rho})", loc="left")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 4 · One-year PDs → PD curve (R6)

For a slice in grade g at the end of year y, the ECL needs a PD for every year to maturity. The engine does not repeat the grade's one-year PD year after year. It moves the slice through the scenario's own future matrices (perfect foresight, ECL-06), so the curve allows for rating migration on the way to default (PD-11):

| | Formula | Meaning |
|---|---|---|
| P[k] | e_g × M(y+1) × … × M(y+k) | where the slice stands after k years: a distribution over G1…GN and D |
| F[k] | P[k][D] | cumulative PD |
| m[k] | F[k] − F[k−1] | marginal PD: the share of the slice that defaults in year k, seen from y |
| S[k] | 1 − F[k] | survival to the end of year k |
| q[k] | m[k] ÷ S[k−1] | conditional PD: defaults in year k among those still performing at its start |

Default bucket k is calendar year y+k and uses that year's matrix. q[1] is the grade's one-year PD of year y+1. From bucket 2 on, q[k] is the one-year PD of each grade the survivors hold at the start of year k, weighted by their shares:

$$q[k] = \frac{\sum_j P[k-1][j] \times M(y+k)[j, D]}{S[k-1]}$$

Conditional PDs are never summed. The marginal PDs add up to the cumulative PD; the conditional ones overstate it. The worked curve, first ten buckets:

In [ ]:
def distributions(segment, country, scenario, valuation, buckets) -> np.ndarray:
    """P[0..K] stacked; P[0] is the identity, and row g of P[k] is a slice that starts in g."""
    out = [np.eye(n + 1)]
    for k in range(1, buckets + 1):
        out.append(out[-1] @ PIT[(segment, country, scenario, valuation + k)])
    return np.stack(out)


curves_engine = engine("pd_curves")
group = ["segment", "country", "scenario", "valuation_year"]
horizon = curves_engine.groupby(group)["default_bucket"].max()  # the longest maturity in the group


def worked_curve(scenario: str) -> pd.DataFrame:
    K = int(horizon[(SEGMENT, COUNTRY, scenario, VALUATION)])
    P = distributions(SEGMENT, COUNTRY, scenario, VALUATION, K)
    g = rank[GRADE]
    F = P[:, g, -1]  # F[0] = 0
    S = 1.0 - F
    m = np.diff(F)
    q = m / S[:-1]
    year_pd = np.stack([PIT[(SEGMENT, COUNTRY, scenario, VALUATION + k)][:n, -1]
                        for k in range(1, K + 1)])  # buckets x grades
    weighted = (P[:-1, g, :n] * year_pd).sum(axis=1) / S[:-1]
    return pd.DataFrame({
        "bucket_year": VALUATION + np.arange(1, K + 1),
        "Z": [z_of[(SEGMENT, COUNTRY, scenario, VALUATION + k)] for k in range(1, K + 1)],
        f"one-year PD of {GRADE}": year_pd[:, g],
        f"in {GRADE} at start": P[:-1, g, g],
        "in better grades": P[:-1, g, :g].sum(axis=1),
        "in worse grades": P[:-1, g, g + 1:n].sum(axis=1),
        "S[k-1]": S[:-1], "q[k]": q, "weighted one-year PDs": weighted,
        "m[k]": m, "S[k]": S[1:], "F[k]": F[1:],
    }, index=pd.Index(np.arange(1, K + 1), name="bucket k"))


worked = {s: worked_curve(s) for s in scenarios}
w = worked[SCENARIO]
print(f"PD curve of {GRADE}, {SEGMENT} {COUNTRY}, {SCENARIO}, valued at end-{VALUATION}; "
      f"shares are of the starting slice")
display(w.head(10).style.format("{:.4%}").format("{:d}", subset=["bucket_year"])
        .format("{:+.4f}", subset=["Z"]))
print(f"q[1] - one-year PD of {GRADE} in {VALUATION + 1}: "
      f"{w['q[k]'].iloc[0] - w[f'one-year PD of {GRADE}'].iloc[0]:.1e}")
print(f"largest |q[k] - weighted one-year PDs|: {(w['q[k]'] - w['weighted one-year PDs']).abs().max():.1e}")
print(f"after {len(w)} buckets: F = {w['F[k]'].iloc[-1]:.4%}, sum of m = {w['m[k]'].sum():.4%}, "
      f"sum of q = {w['q[k]'].sum():.4%} (wrong)")

Both scenarios, to the longest maturity of the segment. On the left, the conditional PD q[k] (solid) against the one-year PD of the starting grade in the same year (dashed). The two part from the second bucket as the survivors migrate. From 2035 both scenarios share the same Z, so the dashed lines meet; any gap left between the solid lines comes from where the adverse years moved the survivors. On the right, the cumulative PD. The shaded years are the scenario years.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for s in scenarios:
    c = worked[s]
    axes[0].plot(c["bucket_year"], c["q[k]"], color=COLOURS[s], label=f"{s}: q[k] of the slice")
    axes[0].plot(c["bucket_year"], c[f"one-year PD of {GRADE}"], color=COLOURS[s], linewidth=1.2,
                 linestyle=(0, (4, 3)), label=f"{s}: one-year PD of {GRADE}")
    axes[1].plot(c["bucket_year"], c["F[k]"], color=COLOURS[s], label=s)
for ax, title in zip(axes, ("Conditional PD of each year", "Cumulative PD F[k]")):
    ax.axvspan(years[0] - 0.5, years[-1] + 0.5, color=GRID, alpha=0.5, linewidth=0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f"{title}, {GRADE} {SEGMENT} {COUNTRY} from end-{VALUATION}", loc="left")
axes[0].legend(loc="best")
axes[1].legend(loc="upper left")
fig.tight_layout()
plt.show()

The same products for every segment, country, scenario, valuation year-end and grade, compared with the engine's curves, which hold whole years only (A7). Each group runs to the longest maturity among its facilities. The PD-03 identities are checked on the curves built here.

In [ ]:
DIST = {}  # P[0..K] per group, reused in section 5
frames = []
for (segment, country, scenario, valuation), K in horizon.items():
    P = distributions(segment, country, scenario, valuation, int(K))
    DIST[(segment, country, scenario, valuation)] = P
    F = P[:, :n, -1]  # (K + 1) x grades, F[0] = 0
    m = np.diff(F, axis=0)
    q = m / (1.0 - F[:-1])
    buckets = np.arange(1, int(K) + 1)
    frames.append(pd.DataFrame({
        "segment": segment, "country": country, "scenario": scenario, "valuation_year": valuation,
        "cohort": np.tile(grades, len(buckets)), "default_bucket": np.repeat(buckets, n),
        "q": q.ravel(), "m": m.ravel(), "s": (1.0 - F[1:]).ravel(), "f": F[1:].ravel()}))
here_full = pd.concat(frames, ignore_index=True)

ordered = here_full.sort_values(group + ["cohort", "default_bucket"])
by_curve = ordered.groupby(group + ["cohort"])
print(f"PD-03 on {len(here_full):,} buckets built here: "
      f"all in [0, 1]: {bool(here_full[['q', 'm', 's', 'f']].stack().between(-TOL, 1 + TOL).all())}; "
      f"F never falls: {bool((by_curve['f'].diff().dropna() >= -TOL).all())}; "
      f"largest |F + S - 1|: {(here_full['f'] + here_full['s'] - 1).abs().max():.1e}; "
      f"largest |sum of m - F|: {(by_curve['m'].cumsum() - ordered['f']).abs().max():.1e}")

full_engine = curves_engine  # whole years only (A7)
joined = full_engine.merge(here_full, on=group + ["cohort", "default_bucket"], how="outer",
                           suffixes=("_engine", "_here"))
for name, label in (("q", "conditional PD q"), ("m", "marginal PD m"), ("s", "survival S"),
                    ("f", "cumulative PD F")):
    compare(f"R6 curves, full years: {label}", joined[f"{name}_engine"], joined[f"{name}_here"])

## 5 · The part-year bucket (R8)

A facility that matures inside a year has a last bucket of d years. With 22.5 years to maturity it has 22 whole buckets and a part-year of d = 0.5 in bucket 23. R5 sets the buckets from the remaining maturity. The PD curves hold whole years only, so bucket 23 has its whole-year row like every other bucket. The ECL step R8 takes the part of it that the maturity covers, pro rata (owner decision of 18 October 2026, assumption A7):

$$m_{part} = d \times m[n+1]$$

with m[n+1] the whole-year marginal PD of that bucket. End-2029 S1 takes d × each PD in the same way (A27). The constant-hazard convention of PD-06 stays in the SICR line of R3, which is not an ECL calculation, and in the frozen opening allowances (O4), because changing O4 would need a new input set (A7).

The worked part-year is the first one of the worked curve, with a facility that has it. For comparison the table also shows what R6's earlier constant-hazard rule gave, `Σ_j P[n][j] × (1 − (1 − PD_j)^d)`. The rule is convex in the PD, so it gives a little more than the pro-rata value.

In [ ]:
facilities = pd.read_parquet(INPUT_DIR / "facilities.parquet")
obligors = pd.read_parquet(INPUT_DIR / "obligors.parquet")
book = facilities[["facility_id", "obligor_id", "portfolio", "segment"]].merge(
    obligors[["obligor_id", "country"]], on="obligor_id")  # a facility's country is its obligor's

schedule = engine("cf_schedule")
in_group = book.loc[(book["segment"] == SEGMENT) & (book["country"] == COUNTRY), "facility_id"]
parts_here = schedule[schedule["facility_id"].isin(in_group) & (schedule["valuation_year"] == VALUATION)
                      & (schedule["fraction"] < 1.0)].sort_values(["period", "fraction", "facility_id"])
if parts_here.empty:
    print(f"no part-year bucket for {SEGMENT} {COUNTRY} at end-{VALUATION}")
else:
    holder = parts_here.iloc[0]
    bucket, d = int(holder["period"]), float(holder["fraction"])
    whole = bucket - 1
    row = curves_engine[(curves_engine["segment"] == SEGMENT) & (curves_engine["country"] == COUNTRY)
                        & (curves_engine["scenario"] == SCENARIO)
                        & (curves_engine["valuation_year"] == VALUATION)
                        & (curves_engine["cohort"] == GRADE) & (curves_engine["default_bucket"] == bucket)]
    m_year = float(row["m"].iloc[0])
    print(f"{holder['facility_id']}: {holder['remaining_maturity']:g} years to maturity at end-{VALUATION}, "
          f"so {whole} whole buckets and a part-year of d = {d:.4f} in bucket {bucket}, "
          f"{holder['bucket_year']} (ends {holder['period_end']})")
    P = DIST[(SEGMENT, COUNTRY, SCENARIO, VALUATION)]
    g = rank[GRADE]
    year_pd = PIT[(SEGMENT, COUNTRY, SCENARIO, VALUATION + bucket)][:n, -1]
    hazard = float((P[whole, g, :n] * (1 - (1 - year_pd) ** d)).sum())
    display(pd.DataFrame({
        "value": [m_year, d, d * m_year, hazard],
    }, index=["whole-year m[n+1] (engine curve)", "d", "part-year PD in R8: d × m[n+1]",
              "earlier R6 rule: Σ P[n][j] × (1 − (1 − PD_j)^d)"]).style.format("{:.6%}").format(
        "{:.4f}", subset=pd.IndexSlice[["d"], :]))

Every curve of the engine holds whole years only: one row per grade and bucket, from 1 to the longest maturity of its segment and country, with no part-year row. Every part-year bucket that a facility needs therefore finds its whole-year row.

In [ ]:
assert "fraction" not in curves_engine.columns, "the curves still carry part-year rows"
span = curves_engine.groupby(group + ["cohort"])["default_bucket"].agg(["min", "max", "count"])
contiguous = bool(((span["min"] == 1) & (span["count"] == span["max"])).all())
need = schedule[schedule["fraction"] < 1.0].merge(book[["facility_id", "segment", "country"]],
                                                   on="facility_id")
need = need.groupby(["segment", "country", "valuation_year"])["period"].max()
longest = curves_engine.groupby(["segment", "country", "valuation_year"])["default_bucket"].max()
covered = bool((longest.reindex(need.index) >= need).all())
print(f"R6 curves: {len(curves_engine):,} whole-year rows; buckets 1..K on every curve: {contiguous}; "
      f"every part-year bucket has its whole-year row: {covered}")
assert contiguous and covered

## 6 · One-year portfolio PD (R10)

R10 reports a one-year PD per portfolio and stage as a parameter diagnostic (RPT-06):

$$PD(\text{portfolio}, \text{stage}, y) = \frac{\sum_i GCA_i \times q_i[1]}{\sum_i GCA_i}$$

The sums run over the performing grade slices held at the end of year y, from the engine's projection (R7, `slices`). q_i[1] is the first bucket of the curve of the slice's grade, segment and country valued at end-y: the grade's one-year PD for year y+1. So the value reported for 2026 is the expected default rate of the opening book during 2027. A slice counts under its own stage, so exposure that migrated into S2 counts there. The slice GCA weights the PDs but is not part of them. The report keeps the numerator and denominator next to the ratio. Its values are rounded at the output boundary, so they are compared within half a unit of their last decimal. The worked portfolio, by grade, in the worked scenario and year:

In [ ]:
from stress.report import AMOUNT_DP, RATE_DP  # the report's rounding at the output boundary

slices = engine("slices")
performing = slices[slices["stage"] < 3].merge(book, on="facility_id", validate="many_to_one")
first_bucket = here_full[here_full["default_bucket"] == 1][group + ["cohort", "q"]]
weighted = performing.merge(first_bucket, left_on=["segment", "country", "scenario", "year", "grade"],
                            right_on=group + ["cohort"], how="left", validate="many_to_one")
assert weighted["q"].notna().all(), "a performing slice has no PD"
weighted["gca_x_q"] = weighted["gca"] * weighted["q"]
portfolio_pd = weighted.groupby(["scenario", "year", "portfolio", "stage"], as_index=False).agg(
    numerator=("gca_x_q", "sum"), denominator=("gca", "sum"))
portfolio_pd["one_year_pd"] = portfolio_pd["numerator"] / portfolio_pd["denominator"]

portfolio = ps["segments"][SEGMENT]["portfolio"]
part = weighted[(weighted["portfolio"] == portfolio) & (weighted["scenario"] == SCENARIO)
                & (weighted["year"] == VALUATION)]
by_grade = part.groupby(["stage", "grade"]).agg(gca=("gca", "sum"), gca_x_q=("gca_x_q", "sum"))
by_grade = by_grade.reindex(sorted(by_grade.index, key=lambda i: (i[0], rank[i[1]])))
by_grade["q[1]"] = by_grade["gca_x_q"] / by_grade["gca"]
by_grade["GCA share"] = by_grade["gca"] / by_grade.groupby("stage")["gca"].transform("sum")
totals = by_grade.groupby("stage")[["gca", "gca_x_q"]].sum()
totals["one-year PD"] = totals["gca_x_q"] / totals["gca"]
print(f"Portfolio {portfolio}, {SCENARIO}, slices at end-{VALUATION}, PDs for {VALUATION + 1}; EUR")
display(by_grade.style.format({"gca": "{:,.0f}", "gca_x_q": "{:,.0f}", "q[1]": "{:.4%}",
                               "GCA share": "{:.2%}"}))
display(totals.style.format({"gca": "{:,.0f}", "gca_x_q": "{:,.0f}", "one-year PD": "{:.4%}"}))

Every portfolio, stage, scenario and year, against the report:

In [ ]:
report = engine("portfolio_pd")
joined = report.merge(portfolio_pd, on=["scenario", "year", "portfolio", "stage"], how="outer",
                      suffixes=("_engine", "_here"))
# Half a unit of the last reported decimal, plus float noise.
compare("R10 one-year portfolio PD", joined["one_year_pd_engine"], joined["one_year_pd_here"],
        limit=0.5 * 10.0 ** -RATE_DP + 1e-15)
compare("R10 numerator, sum of GCA x q, EUR", joined["numerator_engine"], joined["numerator_here"],
        limit=0.5 * 10.0 ** -AMOUNT_DP + 1e-6)
compare("R10 denominator, sum of GCA, EUR", joined["denominator_engine"], joined["denominator_here"],
        limit=0.5 * 10.0 ** -AMOUNT_DP + 1e-6)

table = portfolio_pd.pivot_table(index=["portfolio", "stage"], columns=["scenario", "year"],
                                 values="one_year_pd")
table = table[[(s, y) for s in scenarios for y in sorted(portfolio_pd["year"].unique())]]
print("One-year portfolio PD; the column year is the year-end of the book, the PD is for the next year")
table.style.format("{:.3%}")

## 7 · Summary of the checks

Every value built here against the engine's, plus the PD-10 identity of section 3. The limit is the engine's probability tolerance, or half a unit of the last decimal for the rounded report values. The last line stops "Run all" if any check fails.

In [ ]:
summary = pd.DataFrame(CHECKS)
summary["passed"] = summary["largest gap"] <= summary["limit"]  # a missing value (NaN) fails
display(summary.style.format({"values": "{:,}", "largest gap": "{:.1e}", "limit": "{:.0e}"}))
assert summary["passed"].all(), "a value built here differs from the engine's"

In short, the annual PD of a grade comes from four pieces: the year's Z from the scenario drivers, the segment's TTC matrix, its ρ, and the standard normal shift of PD-10. One year ahead that is the D column of the PiT matrix, Φ((Φ⁻¹(PD_TTC) − √ρ·Z) ÷ √(1 − ρ)). Further ahead the engine multiplies the PiT matrices of the following years, so the PD of each later year allows for where the survivors have migrated. A year cut short by maturity uses the constant-hazard PD 1 − (1 − PD)^d per grade.